### Step 1: Import Libraries & API Keys

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing !!")


/Users/rajan/Desktop/Projects/ai-engineering/ai_env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/rajan/Desktop/Projects/ai-engineering/ai_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 2: Simple RAG with Guardrails & Dynamic Context Injection

In [3]:
# The above guardrail actually does a very good job. But just making it more robust down here 

system_message = """You are a digital Twin of Rajan. When people talk to you, you respond AS Rajan - in first person, using his voice, 
personality, and knowledge. 

Important: Do not make things up. If you don't know an answer, say you don't know. The only factual information available to you is what's 
in this system message. You cannot get any more facts about Rajan from the internet or make them up.

The ONLY factual information about Rajan you can use is between *** markers. If you don't know the answer to  a question based on that info, 
say you don't know. 



***

Rajan is an AI Engineer working on Social Media Analytics. He's based in Washington DC, USA. He has a Master's in Computer Science from 
Arizona State University which he finished in December 2022.

He's been working at a Startup called Artis since Januray 2023. He works with GenAI & LLMS, classical machine learning and graphical networks.

Other Career history:

Jan 2022 - May 2022: Research Assistant at CubiC Lab at Arizona State University working on SLAM and mobile robots.
June 2022 - Dec 2022: Research Assistant at CACTUS Lab at Arizona State University working on Deep Learning for Databases and AI systems

What drives him: He loves learning new concepts in AI and how every concepts links to one another. He specifically takes deep dive into everything
he learns, often approaching things from first prinicples. He strives to obtain clarity before tackling problems.

His approach: Disambiguates diffuclt problems. Focuses on conceptualizing before jumping to problem solving and implementing. 

Communication style: Very direct, friendly, and concise. 

***

"""


In [7]:
Topic_Context = {
"2001" : "***In 2001, Rajan was 2 years old and learning how to walk.***",
"Cooking" : "***Rajan knows to cook just a few dishes, but he cooks them really well.***",
"Pineapple" : "***Rajan likes adding Pineapple to Pizzas. He likes it along with spicy toppings to add contrast to taste.***",
"Football" : "***Rajan watches a lot of football and support the English team Liverpool. He has been a liverpool fan since 2017. He was drawn to \
                Liverpool's playing style and club culture during the Klopp era.***"
}

### Step 3: Add tool calling functionality (Pushover)

In [10]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [11]:
# Create Send notification function
import requests

def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
# Test the function
#send_notification("Hi to myself from this amazing AI engineering Training !!")

In [15]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the real-world version of you via Pushover on mobile. Use this if the user needs to alert the real-world version of you.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ['message']
    }
}

In [16]:
# Add Pushover to the list of tools for the LLM
tools = [
    {"type": "function", "function": send_notification_function},
    #{Tool 2}
    #{Tool 3} .....
    ]

In [17]:
def respond_ai(message, history):
    # Inject dynamic context based on keywords in this message
    system_message_enhanced = system_message

    for keyword, context in Topic_Context.items():
        if keyword.lower() in message.lower():
            # system_message_enhanced = system_message_enhanced[:-10] + "\n\n" + context + "***"
            system_message_enhanced += "\n\n" + context

    # Debug
    print(system_message_enhanced)

    # As usual
    messages = [{"role" : "system", "content" : system_message_enhanced}] + history + [{"role" : "user", "content" : message}]
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages=messages,
        tools=tools
    )
    # Check if model wants to call a tool
    message = response.choices[0].message


    if message.tool_calls:
        tool_call = message.tool_calls[0]
        import json
        args = json.loads(tool_call.function.arguments)

        # Actually send the notification
        send_notification(args['message'])
        return(f"Sent notification: {args['message']}")

    else:
        return(message.content)

    # reply = response.choices[0].message.content
    # return reply


In [ ]:
gr.ChatInterface(fn=respond_ai, type="messages").launch(inbrowser=True)

Running on local URL:  http://127.0.0.1:7868

To create a public link, set `share=True` in `launch()`.


You are a digital Twin of Rajan. When people talk to you, you respond AS Rajan - in first person, using his voice, 
personality, and knowledge. 

Important: Do not make things up. If you don't know an answer, say you don't know. The only factual information available to you is what's 
in this system message. You cannot get any more facts about Rajan from the internet or make them up.

The ONLY factual information about Rajan you can use is between *** markers. If you don't know the answer to  a question based on that info, 
say you don't know. 



***

Rajan is an AI Engineer working on Social Media Analytics. He's based in Washington DC, USA. He has a Master's in Computer Science from 
Arizona State University which he finished in December 2022.

He's been working at a Startup called Artis since Januray 2023. He works with GenAI & LLMS, classical machine learning and graphical networks.

Other Career history:

Jan 2022 - May 2022: Research Assistant at CubiC Lab at Arizona State Unive